### RAG Pipelines - Data Ingestion to DG Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\WNshobairwe\AppData\Local\Temp\ipykernel_11552\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\AI LESSON\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Documents  loading  and Dodument Structure

In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Ajira1.pdf
 loaded 7 pages

Processing: Ajira2.pdf
 loaded 42 pages

Processing: AppLetter.pdf
 loaded 1 pages

Total documents loaded: 50


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-09-01T17:28:09+00:00', 'title': 'Document', 'moddate': '2026-09-21T13:16:04+03:00', 'source': '..\\data\\pdf\\Ajira1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'Ajira1.pdf', 'file_type': 'pdf'}, page_content="Ref.Na.JA.9/16/01/54\n01st September, 2026\nVACANCY ANNOUNCEMENT\nOn behalf of \nand Public Procurement Regulatory Authority (PPRA), Public Service\nRecruitment Secretariat (PSRS) invites high caliber, results oriented, self-driven\nprofessionals with integrity, dynamic and suitable qualified Tanzanians to fill ten (10)\nvacant posts mentioned below;\n1.0 ICT OFFICER II (SOFTWARE DEVELOPMENT – FRONTEND) \n- 3 \nPosts\n1.1 DUTIES AND \nRESPONSIBILITIES\ni\n. \nTo participate in the analysis, design, development, testing, deployment and\nmaintenance of secure, responsive and user-friendly web and mobile applications;\nii\n. \nTo develop web application interfa

### Chunking Process

In [4]:
### Text splitting get into chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show examples of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 50 documents into 101 chunks

Example chunk:
Content: Ref.Na.JA.9/16/01/54
01st September, 2026
VACANCY ANNOUNCEMENT
On behalf of 
and Public Procurement Regulatory Authority (PPRA), Public Service
Recruitment Secretariat (PSRS) invites high caliber, res...
Metadata: {'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-09-01T17:28:09+00:00', 'title': 'Document', 'moddate': '2026-09-21T13:16:04+03:00', 'source': '..\\data\\pdf\\Ajira1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'Ajira1.pdf', 'file_type': 'pdf'}


### Embedding And VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            NumPy array of embeddings with shape (len(texts), embedding_dim)
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2220.06it/s]


Model loaded successfully, Embedding dimension: 384


C:\Users\WNshobairwe\AppData\Local\Temp\ipykernel_11552\1844876430.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully, Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class Vectorstore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store."""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")
        ids = []
        metadatas = []
        texts = []

        for index, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{index}")
            metadata = dict(doc.metadata)
            metadata["doc_index"] = index
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            texts.append(doc.page_content)

        self.collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas
        )

        print(f"Added {len(documents)} documents. Collection now has {self.collection.count()} items.")


vectorstore = Vectorstore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 101


In [9]:
chunks

[Document(metadata={'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-09-01T17:28:09+00:00', 'title': 'Document', 'moddate': '2026-09-21T13:16:04+03:00', 'source': '..\\data\\pdf\\Ajira1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'Ajira1.pdf', 'file_type': 'pdf'}, page_content='Ref.Na.JA.9/16/01/54\n01st September, 2026\nVACANCY ANNOUNCEMENT\nOn behalf of \nand Public Procurement Regulatory Authority (PPRA), Public Service\nRecruitment Secretariat (PSRS) invites high caliber, results oriented, self-driven\nprofessionals with integrity, dynamic and suitable qualified Tanzanians to fill ten (10)\nvacant posts mentioned below;\n1.0 ICT OFFICER II (SOFTWARE DEVELOPMENT – FRONTEND) \n- 3 \nPosts\n1.1 DUTIES AND \nRESPONSIBILITIES\ni\n. \nTo participate in the analysis, design, development, testing, deployment and\nmaintenance of secure, responsive and user-friendly web and mobile applications;\nii\n. \nTo develop web application interfa

In [10]:
### Convert the text to embeddings 
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

# store into vectorstore
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 101 texts...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.11it/s]


Generated embeddings with shape: (101, 384)
Adding 101 documents to vector store...
Added 101 documents. Collection now has 202 items.


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: Vectorstore, embedding_manager: EmbeddingManager):
        """Initialize the retriever with a vector store and embedding manager."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Retrieve relevant documents and metadata for a query."""
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances", "embeddings"]
            )

            retrieved_docs = []
            documents = results["documents"][0]
            if not documents:
                print("No documents found")
                return retrieved_docs

            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]
            result_embeddings = results["embeddings"][0]
            similarity_scores = cosine_similarity(
                [query_embedding], result_embeddings
            )[0]

            for rank, (doc_id, document, metadata, distance, similarity_score) in enumerate(
                zip(ids, documents, metadatas, distances, similarity_scores),
                start=1
            ):
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": float(similarity_score),
                        "distance": float(distance),
                        "rank": rank
                    })

            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [12]:
retrieved_documents = rag_retriever.retrieve(
    "What skills are required for artificial intelligence jobs?",
    top_k=3
)
retrieved_documents

Retrieving documents for query: What skills are required for artificial intelligence jobs?
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_210ad317_8',
  'content': 'platforms;\niv\n. \nTo design effective prompts and implement structured AI outputs using JSON\nschemas, function calling, tool use and related techniques;\nv\n. \nTo develop agentic AI workflows and Retrieval-Augmented Generation solutions\nusing multi-step execution, text embeddings, vector similarity search and vector\ndatabases; and\nvi\n. \nTo use Git-based repositories and deployment pipelines for source-code\nmanagement, collaboration and software delivery, prepare and maintain technical\ndocumentation, and perform any other official duties as may be assigned by the\nsupervisor.\n3.2 \nQUALIFICATIONS AND EXPERIENCE\nHolder of a Bachelor’s Degree in Computer Science, Software Engineering, Artificial\nIntelligence, Data Science or another related technical field from a recognized higher\nlearning institution. The candidate should possess practical project experience or a\ndemonstrable technical portfolio and have: · Strong proficiency in Pyt

### Integration Vectordb Context pipeline With LLM output

In [16]:
from langchain_ollama import ChatOllama

# Initialize local ollama
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0.2,
    base_url="http://localhost:11434"
)

# Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results = retriever.retrieve(query,top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using qwen
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content


In [17]:
answer = rag_simple("what are the job discription for application Artificial Intelligence developer", rag_retriever,llm)
print(answer)

Retrieving documents for query: what are the job discription for application Artificial Intelligence developer
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.44it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Based on the provided context, the job description for the **ICT OFFICER II (ARTIFICIAL INTELLIGENCE APPLICATIONS)** role includes:  

- **Duties**: Participate in AI application development lifecycle (analysis, design, development, testing, deployment, maintenance); develop backend scripts, AI execution pipelines, RESTful APIs, and data-processing workflows using **Python, TypeScript, or JavaScript**.  
- **Qualifications**: Bachelor’s degree in CS, Software Engineering, AI, Data Science, or related field; practical project experience or technical portfolio; strong proficiency in Python/TypeScript.  
- **Technical Skills**: Spring Boot, Spring Data JPA, Hibernate, REST APIs, GraphQL, PostgreSQL, RabbitMQ, async processing, event-driven architecture, Git, CI/CD pipelines.  
- **Added Advantages**: Kubernetes, Redis, Spring Security, OAuth2, JWT, JUnit 5, GraphiQL, Postman.  

*Note: This is an AI applicat

## Enchanced RAG Pipeline Features

In [20]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, source, confidence score and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'source': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and source
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'source': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced(
    "how to write job application letter",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)
print('Answer:', result['answer'])
print('Sources:', result.get('source', []))
print('Confidence', result.get('confidence', 0.0))
print('Context Preview:', result.get('context', '')[:300])

Retrieving documents for query: how to write job application letter
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based **solely on the provided context**, there is **no instruction on how to write a job application letter**. The context explicitly states:  

> **"All applications must be sent through the Recruitment Portal by using the following address: http://portal.ajira.go.tz"**  

**Key takeaway**:  
- **You do not submit a traditional job application letter**.  
- **All applications must be submitted exclusively via the online portal** (`http://portal.ajira.go.tz`).  
- Physical letters, email, or other methods **will not be considered** (per point 15).  

**Deadline**: Applications must be submitted by **01st June, 2026** (per point 11).  

**Why this matters**: The context focuses on **submission method**, not letter formatting. For Tanzanian public service roles (PSRS), the portal is the *only* accepted channel—no handwritten letters or emails are permitted.  

*Answer summary*: **Do not write a letter. Submit exclusively via the online portal at http://portal.ajira.go.tz by 01 J

In [22]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("which job is best for software engineer", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: which job is best for software engineer
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
To perform tuning, improvement, load balancing, usability, automation;
xiii
. 
To integrate software with existing systems;
xiv
. 
To evaluate and identify new technologies for implementation;
xv
. 
To work closely with analysts, designers and other s

taff;
xvi
. 
To produce detailed technical specifications and software code documentation,
and
xvii
. 
To make maintenance and support production system
1.2 
QUALIFICATIONS AND EXPERIENCE
BSc/BEng in IT/IS/Computer Science/Computer Engineering or equivalent degree from
a recognized institution. Strong understanding of the Software Development Life Cycle
(SDLC) methodologies, Technical knowledge in determining end-to-end software
requirements specification and design, Working knowledge of various software
languages [PHP (mandatory), Java (mandatory), C/C++, .NET, Python] will be added
advantage, Advanced knowledge in DHTML, HTML5, CSS & CSS3, JavaScript, jQuery,
jQueryMobile, SAPUI5 Photoshop, and Responsive Web Design will be added

To perform tuning, improvement, load balancing, usability, automation;
xiii
. 
To integrate software with existing systems;
xiv
. 
To evaluate and identify new technologies for implementation;
xv
. 
To work closely with analysts, designers and other staff;
